# Step 1D: NMF Consensus Clustering for Patient Stratification
Discovers molecular subtypes in AD/MCI patients using non-negative matrix factorization (NMF).

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import NMF
from sklearn.metrics import silhouette_score, pairwise_distances
from scipy.cluster.hierarchy import cophenet, linkage
from scipy.spatial.distance import pdist
from scipy.stats import kruskal, mannwhitneyu, chi2_contingency
import gseapy as gp
import warnings
import os
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.facecolor'] = 'white'

## Configuration

In [ ]:
# Paths
PROCESSED_DATA_DIR = "../data/processed"
RESULTS_DIR = "../results/step1"

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Parameters
K_RANGE = range(2, 9)  # Try 2 to 8 subtypes
N_CONSENSUS_RUNS = 50  # 50 random initializations
COPHENETIC_THRESHOLD = 0.85  # Min cophenetic correlation
MIN_CLUSTER_SIZE = 25  # Min patients per cluster
TOP_PROTEINS_PER_SUBTYPE = 50

print("Configuration loaded.")

## Generate Test Data (if needed)

In [ ]:
import os

# Check if master table exists
if not os.path.exists(f"{PROCESSED_DATA_DIR}/master_patient_table.csv"):
    print("[SETUP] Generating synthetic data...\n")
    
    np.random.seed(42)
    
    # Generate master table
    n_patients = 180
    patient_ids = [f'patient_{i:03d}' for i in range(n_patients)]
    
    # Diagnosis: ~50% controls (cogdx=1), ~25% MCI (cogdx=2), ~25% AD (cogdx=3-4)
    cogdx = np.random.choice([1, 2, 3], size=n_patients, p=[0.5, 0.25, 0.25])
    
    master_df = pd.DataFrame({
        'diagnosis': cogdx,
        'cogdx': cogdx,
        'braaksc': np.clip(cogdx * 1.5 + np.random.normal(0, 0.5, n_patients), 0, 6).astype(int),
        'ceradsc': np.clip(5 - cogdx + np.random.normal(0, 0.3, n_patients), 1, 4).astype(int),
        'mmse': np.clip(30 - (cogdx - 1) * 5 + np.random.normal(0, 2, n_patients), 0, 30).astype(int),
        'age_death': np.random.randint(60, 100, n_patients),
        'msex': np.random.choice([0, 1], n_patients),
        'pmi': np.random.uniform(2, 30, n_patients),
        'ct_Ex': np.random.uniform(0, 1, n_patients),
        'ct_In': np.random.uniform(0, 1, n_patients),
        'ct_Ast': np.random.uniform(0, 1, n_patients),
        'ct_Oli': np.random.uniform(0, 1, n_patients),
        'ct_Mic': np.random.uniform(0, 1, n_patients),
        'ct_OPCs': np.random.uniform(0, 1, n_patients),
        'dpt_pseudotime': np.random.uniform(0, 1, n_patients),
        'umap_1': np.random.normal(0, 1, n_patients),
        'umap_2': np.random.normal(0, 1, n_patients)
    }, index=patient_ids)
    
    # Normalize cell-type proportions
    ct_cols = ['ct_Ex', 'ct_In', 'ct_Ast', 'ct_Oli', 'ct_Mic', 'ct_OPCs']
    master_df[ct_cols] = master_df[ct_cols].div(master_df[ct_cols].sum(axis=1), axis=0)
    
    master_df.index.name = 'projid'
    master_df.to_csv(f"{PROCESSED_DATA_DIR}/master_patient_table.csv")
    
    # Generate deconvolved profiles
    n_proteins = 1500
    protein_ids = [f'PROT_{i:05d}' for i in range(n_proteins)]
    
    # Create deconvolved profiles: (patient, protein, cell_type)
    deconvolved_data = []
    for i, pid in enumerate(patient_ids):
        for j, protein in enumerate(protein_ids):
            for k, ct in enumerate(ct_cols):
                abundance = np.random.exponential(1.0) * master_df.loc[pid, ct] + np.random.normal(0, 0.1)
                abundance = max(0, abundance)  # Ensure non-negative
                deconvolved_data.append({
                    'sample_id': pid,
                    'protein_id': protein,
                    'cell_type': ct,
                    'abundance': abundance
                })
    
    deconvolved_df = pd.DataFrame(deconvolved_data)
    deconvolved_df.to_csv(f"{PROCESSED_DATA_DIR}/deconvolved_profiles.csv", index=False)
    
    # Generate bulk proteomics
    bulk_data = np.random.exponential(1.0, size=(n_patients, n_proteins))
    bulk_df = pd.DataFrame(bulk_data, index=patient_ids, columns=protein_ids)
    bulk_df.index.name = 'patient_id'
    bulk_df.to_csv(f"{PROCESSED_DATA_DIR}/rosmap_proteomics_cleaned.csv")
    
    print(f"  Generated: {PROCESSED_DATA_DIR}/master_patient_table.csv ({master_df.shape})")
    print(f"  Generated: {PROCESSED_DATA_DIR}/deconvolved_profiles.csv ({deconvolved_df.shape})")
    print(f"  Generated: {PROCESSED_DATA_DIR}/rosmap_proteomics_cleaned.csv ({bulk_df.shape})")
    print(f"  Cogdx distribution: {np.bincount(cogdx)}\n")
else:
    print("[INFO] Found existing data files.\n")

## Load Data

In [ ]:
def load_data(processed_dir):
    """
    Load master table and proteomics data.
    """
    print("[1] Loading data...")
    
    master_df = pd.read_csv(f"{processed_dir}/master_patient_table.csv", index_col=0)
    bulk_df = pd.read_csv(f"{processed_dir}/rosmap_proteomics_cleaned.csv", index_col=0)
    
    print(f"  Master table: {master_df.shape}")
    print(f"  Bulk proteomics: {bulk_df.shape}")
    print(f"  Cogdx distribution: {master_df['cogdx'].value_counts().sort_index().to_dict()}")
    
    return master_df, bulk_df

In [ ]:
def prepare_clustering_data(master_df, bulk_df, processed_dir):
    """
    Prepare data for clustering: select AD/MCI patients, load deconvolved profiles.
    """
    print("\n[2] Preparing clustering data...")
    
    # Select AD/MCI patients (cogdx >= 2)
    ad_mci_mask = master_df['cogdx'] >= 2
    ad_mci_patients = master_df[ad_mci_mask].index.tolist()
    
    print(f"  AD/MCI patients: {len(ad_mci_patients)} out of {len(master_df)}")
    print(f"  Cogdx in clustering set: {master_df.loc[ad_mci_patients, 'cogdx'].value_counts().sort_index().to_dict()}")
    
    # Try to load deconvolved profiles
    try:
        deconvolved_df = pd.read_csv(f"{processed_dir}/deconvolved_profiles.csv")
        
        # Pivot to matrix: rows=patients, columns=protein-celltype combinations
        deconvolved_pivot = deconvolved_df.pivot_table(
            index='sample_id',
            columns=['protein_id', 'cell_type'],
            values='abundance',
            fill_value=0
        )
        
        # Flatten column names
        deconvolved_pivot.columns = [f"{p}_{ct}" for p, ct in deconvolved_pivot.columns]
        
        # Subset to AD/MCI patients
        clustering_matrix = deconvolved_pivot.loc[ad_mci_patients].fillna(0)
        
        print(f"  Deconvolved profiles shape: {clustering_matrix.shape}")
        print(f"  Features: {clustering_matrix.shape[1]} (protein-celltype combinations)")
        
        data_source = 'deconvolved'
    except Exception as e:
        print(f"  Could not load deconvolved profiles: {e}")
        print(f"  Falling back to bulk proteomics...")
        
        clustering_matrix = bulk_df.loc[ad_mci_patients].fillna(0)
        print(f"  Bulk proteomics shape: {clustering_matrix.shape}")
        
        data_source = 'bulk'
    
    return clustering_matrix, ad_mci_patients, data_source

In [ ]:
def ensure_nonnegative(matrix):
    """
    Ensure all values are non-negative for NMF.
    If minimum is negative, shift all values.
    """
    print("\n[3] Ensuring non-negative values for NMF...")
    
    min_val = matrix.min().min()
    
    if min_val < 0:
        print(f"  Min value: {min_val:.4f} (negative)")
        matrix = matrix - min_val + 1e-6  # Shift and add small epsilon
        print(f"  Shifted matrix to non-negative range")
    else:
        print(f"  Min value: {min_val:.4f} (already non-negative)")
    
    return matrix

In [ ]:
def run_nmf_consensus(clustering_matrix, k_range=range(2, 9), n_runs=50):
    """
    Run NMF consensus clustering across k values.
    Returns: dictionary with k as key, clustering results as value.
    """
    print(f"\n[4] Running NMF consensus clustering (k={min(k_range)}-{max(k_range)}, {n_runs} runs per k)...")
    
    n_samples = clustering_matrix.shape[0]
    results = {}
    
    for k in k_range:
        print(f"\n  k={k}:")
        
        # Initialize consensus matrix
        consensus_matrix = np.zeros((n_samples, n_samples))
        
        # Run NMF once with nndsvda initialization
        nmf_base = NMF(n_components=k, init='nndsvda', max_iter=1000, random_state=42)
        H_base = nmf_base.fit_transform(clustering_matrix)
        labels_base = np.argmax(H_base, axis=1)
        
        # Run NMF 50 times with random initialization
        all_labels = [labels_base]  # Include base solution
        
        for run in range(n_runs):
            nmf = NMF(n_components=k, init='random', max_iter=1000, random_state=run)
            H = nmf.fit_transform(clustering_matrix)
            labels = np.argmax(H, axis=1)
            all_labels.append(labels)
            
            if (run + 1) % 10 == 0:
                print(f"    Completed {run+1}/{n_runs} runs")
        
        # Build consensus matrix
        for i in range(n_samples):
            for j in range(n_samples):
                # Count how many runs had i and j in same cluster
                same_cluster = sum(1 for labels in all_labels if labels[i] == labels[j])
                consensus_matrix[i, j] = same_cluster / len(all_labels)
        
        # Compute clustering metrics
        # Convert consensus matrix to distance
        consensus_dist = 1 - consensus_matrix
        np.fill_diagonal(consensus_dist, 0)
        
        # Cophenetic correlation
        condensed_dist = pdist(clustering_matrix, metric='euclidean')
        Z = linkage(condensed_dist, method='average')
        coph_corr, _ = cophenet(Z, condensed_dist)
        
        # Use consensus labels for silhouette (average of all runs)
        consensus_labels = np.round(np.mean(all_labels, axis=0)).astype(int)
        sil_score = silhouette_score(clustering_matrix, consensus_labels, metric='euclidean')
        
        # Check cluster sizes
        unique, counts = np.unique(consensus_labels, return_counts=True)
        min_cluster_size = counts.min()
        
        print(f"    Cophenetic correlation: {coph_corr:.4f}")
        print(f"    Silhouette score: {sil_score:.4f}")
        print(f"    Cluster sizes: {dict(zip(unique, counts))}")
        print(f"    Min cluster size: {min_cluster_size}")
        
        results[k] = {
            'consensus_matrix': consensus_matrix,
            'consensus_labels': consensus_labels,
            'cophenetic': coph_corr,
            'silhouette': sil_score,
            'min_cluster_size': min_cluster_size,
            'H_base': H_base
        }
    
    return results

In [ ]:
def select_optimal_k(results, cophenetic_threshold=0.85, min_cluster_size=25):
    """
    Select optimal k based on cophenetic correlation and cluster size.
    """
    print("\n[5] Selecting optimal k...")
    print(f"  Criteria: cophenetic > {cophenetic_threshold} AND min_cluster_size >= {min_cluster_size}")
    
    optimal_k = None
    
    for k in sorted(results.keys()):
        coph = results[k]['cophenetic']
        min_size = results[k]['min_cluster_size']
        
        print(f"  k={k}: cophenetic={coph:.4f}, min_size={min_size} ", end="")
        
        if coph > cophenetic_threshold and min_size >= min_cluster_size:
            print("[PASS]")
            optimal_k = k
            break
        else:
            print("[FAIL]")
    
    if optimal_k is None:
        # Fallback: pick k with highest cophenetic
        optimal_k = max(results.keys(), key=lambda k: results[k]['cophenetic'])
        print(f"\n  No k passes criteria. Selecting k={optimal_k} with highest cophenetic.")
    
    print(f"\n  Selected k: {optimal_k}")
    
    return optimal_k

In [ ]:
def plot_cluster_selection(results, results_dir):
    """
    Plot cophenetic correlation and silhouette score vs k.
    """
    print("\n[6] Plotting cluster selection metrics...")
    
    k_values = sorted(results.keys())
    coph_values = [results[k]['cophenetic'] for k in k_values]
    sil_values = [results[k]['silhouette'] for k in k_values]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Cophenetic correlation
    ax1.plot(k_values, coph_values, 'o-', linewidth=2, markersize=8, color='steelblue')
    ax1.axhline(y=0.85, color='red', linestyle='--', label='Threshold (0.85)')
    ax1.set_xlabel('Number of Subtypes (k)', fontsize=11)
    ax1.set_ylabel('Cophenetic Correlation', fontsize=11)
    ax1.set_title('Consensus Stability', fontsize=12, fontweight='bold')
    ax1.set_xticks(k_values)
    ax1.grid(alpha=0.3)
    ax1.legend()
    ax1.set_ylim([0.7, 1.0])
    
    # Silhouette score
    ax2.plot(k_values, sil_values, 'o-', linewidth=2, markersize=8, color='coral')
    ax2.set_xlabel('Number of Subtypes (k)', fontsize=11)
    ax2.set_ylabel('Silhouette Score', fontsize=11)
    ax2.set_title('Cluster Separation', fontsize=12, fontweight='bold')
    ax2.set_xticks(k_values)
    ax2.grid(alpha=0.3)
    ax2.set_ylim([min(sil_values) - 0.1, max(sil_values) + 0.1])
    
    plt.tight_layout()
    fig_file = f"{results_dir}/cluster_selection.png"
    plt.savefig(fig_file, dpi=300, bbox_inches='tight')
    print(f"  Saved: {fig_file}")
    plt.close()

In [ ]:
def get_top_proteins_per_subtype(clustering_matrix, H_matrix, top_n=50):
    """
    Get top proteins per subtype from NMF basis vectors.
    Returns: dict with subtype -> list of top protein indices
    """
    print("\n[7] Identifying top proteins per subtype...")
    
    k = H_matrix.shape[0]  # Number of subtypes
    n_proteins = clustering_matrix.shape[1]
    
    top_proteins = {}
    
    for subtype in range(k):
        # Get loadings for this subtype
        loadings = H_matrix[subtype, :]
        
        # Get top proteins
        top_idx = np.argsort(loadings)[::-1][:top_n]
        top_proteins[subtype] = top_idx
        
        print(f"  Subtype {subtype}: identified {len(top_idx)} top proteins")
    
    return top_proteins

In [ ]:
def run_go_enrichment(clustering_matrix, top_proteins_dict):
    """
    Run Gene Ontology enrichment for top proteins per subtype.
    """
    print("\n[8] Running GO enrichment analysis...")
    
    # Map protein indices to names
    protein_names = clustering_matrix.columns.tolist()
    
    enrichment_results = {}
    
    for subtype, top_idx in top_proteins_dict.items():
        top_protein_names = [protein_names[i] for i in top_idx]
        
        # Extract base protein names (remove _celltype suffix if present)
        base_names = [name.split('_')[0] for name in top_protein_names]
        
        print(f"  Subtype {subtype}: Enriching {len(base_names)} proteins...")
        
        try:
            # Run enrichment
            results = gp.enrichr(
                gene_list=base_names,
                gene_sets='GO_Biological_Process_2023',
                organism='Human',
                outdir=None  # Don't save files
            )
            
            # Get top 5 terms
            top_terms = results.results.head(5)[['Term', 'Adjusted P-value']].to_dict(orient='records')
            enrichment_results[subtype] = top_terms
            
            for i, term in enumerate(top_terms[:3]):
                print(f"    {i+1}. {term['Term'][:50]}... (p={term['Adjusted P-value']:.2e})")
        
        except Exception as e:
            print(f"    Error running enrichment: {e}")
            enrichment_results[subtype] = []
    
    return enrichment_results

In [ ]:
def clinical_validation(master_df, ad_mci_patients, labels, results_dir):
    """
    Validate subtypes against clinical measures.
    Compute mean ± SD per subtype, test for differences.
    """
    print("\n[9] Clinical validation of subtypes...")
    
    # Add subtype labels to data
    clinical_data = master_df.loc[ad_mci_patients].copy()
    clinical_data['subtype'] = labels
    
    k = len(np.unique(labels))
    
    # Clinical/cell-type variables to test
    test_vars = ['mmse', 'braaksc', 'ceradsc', 'age_death', 'dpt_pseudotime',
                 'ct_Ex', 'ct_In', 'ct_Ast', 'ct_Oli', 'ct_Mic', 'ct_OPCs']
    
    validation_results = {}
    
    print("\n  Mean ± SD per subtype:")
    print("  " + "-" * 80)
    
    for var in test_vars:
        if var not in clinical_data.columns:
            continue
        
        # Compute mean ± SD per subtype
        stats = {}
        for subtype in range(k):
            subtype_data = clinical_data[clinical_data['subtype'] == subtype][var].dropna()
            stats[subtype] = {
                'mean': subtype_data.mean(),
                'std': subtype_data.std(),
                'n': len(subtype_data)
            }
        
        # Kruskal-Wallis test
        groups = [clinical_data[clinical_data['subtype'] == i][var].dropna().values for i in range(k)]
        stat, pval = kruskal(*groups)
        
        validation_results[var] = {'stats': stats, 'pval': pval}
        
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
        print(f"  {var:20s}: p={pval:.2e} {sig}")
    
    return validation_results

In [ ]:
def save_subtype_labels(subtype_labels, ad_mci_patients, processed_dir):
    """
    Save subtype labels to file.
    """
    print("\n[10] Saving subtype labels...")
    
    labels_df = pd.DataFrame({
        'projid': ad_mci_patients,
        'subtype_label': [f'ST{label+1}' for label in subtype_labels]
    })
    labels_df = labels_df.set_index('projid')
    
    labels_file = f"{processed_dir}/subtype_labels.csv"
    labels_df.to_csv(labels_file)
    print(f"  Saved: {labels_file}")
    
    return labels_df

In [ ]:
def create_master_final(master_df, labels_df, processed_dir):
    """
    Create final master table with subtype labels.
    """
    print("\n[11] Creating final master table...")
    
    master_final = master_df.copy()
    
    # Add subtype labels (controls remain unlabeled)
    master_final['subtype'] = 'Control'
    for patient in labels_df.index:
        master_final.loc[patient, 'subtype'] = labels_df.loc[patient, 'subtype_label']
    
    master_file = f"{processed_dir}/master_patient_table_final.csv"
    master_final.to_csv(master_file)
    print(f"  Saved: {master_file}")
    print(f"  Shape: {master_final.shape}")
    
    return master_final

In [ ]:
def plot_clinical_validation(master_df, ad_mci_patients, labels, results_dir):
    """
    Box plots of clinical measures per subtype.
    """
    print("\n[12] Plotting clinical validation...")
    
    clinical_data = master_df.loc[ad_mci_patients].copy()
    clinical_data['subtype'] = [f'ST{label+1}' for label in labels]
    
    test_vars = ['mmse', 'braaksc', 'dpt_pseudotime']
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    for idx, var in enumerate(test_vars):
        if var not in clinical_data.columns:
            continue
        
        ax = axes[idx]
        
        # Box plot
        data_to_plot = [clinical_data[clinical_data['subtype'] == f'ST{i+1}'][var].dropna().values 
                        for i in range(len(np.unique(labels)))]
        
        bp = ax.boxplot(data_to_plot, patch_artist=True, labels=[f'ST{i+1}' for i in range(len(np.unique(labels)))])
        
        for patch in bp['boxes']:
            patch.set_facecolor('lightblue')
        
        ax.set_ylabel(var, fontsize=11)
        ax.set_xlabel('Subtype', fontsize=11)
        ax.set_title(f'{var.upper()}', fontsize=12, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    fig_file = f"{results_dir}/subtype_clinical.png"
    plt.savefig(fig_file, dpi=300, bbox_inches='tight')
    print(f"  Saved: {fig_file}")
    plt.close()

In [ ]:
def plot_celltype_composition(master_df, ad_mci_patients, labels, results_dir):
    """
    Stacked bar plot of mean cell-type proportions per subtype.
    """
    print("\n[13] Plotting cell-type composition...")
    
    clinical_data = master_df.loc[ad_mci_patients].copy()
    clinical_data['subtype'] = labels
    
    ct_cols = ['ct_Ex', 'ct_In', 'ct_Ast', 'ct_Oli', 'ct_Mic', 'ct_OPCs']
    ct_labels = ['Ex', 'In', 'Ast', 'Oli', 'Mic', 'OPCs']
    
    k = len(np.unique(labels))
    
    # Compute mean proportions per subtype
    mean_props = np.zeros((k, len(ct_cols)))
    
    for subtype in range(k):
        subtype_data = clinical_data[clinical_data['subtype'] == subtype][ct_cols]
        mean_props[subtype, :] = subtype_data.mean().values
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x = np.arange(k)
    colors = plt.cm.Set3(np.linspace(0, 1, len(ct_cols)))
    
    bottom = np.zeros(k)
    for i, ct in enumerate(ct_labels):
        ax.bar(x, mean_props[:, i], bottom=bottom, label=ct, color=colors[i], alpha=0.8)
        bottom += mean_props[:, i]
    
    ax.set_xlabel('Subtype', fontsize=12)
    ax.set_ylabel('Mean Cell-Type Proportion', fontsize=12)
    ax.set_title('Cell-Type Composition per Subtype', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'ST{i+1}' for i in range(k)])
    ax.legend(title='Cell Type', loc='upper right')
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    fig_file = f"{results_dir}/subtype_celltype.png"
    plt.savefig(fig_file, dpi=300, bbox_inches='tight')
    print(f"  Saved: {fig_file}")
    plt.close()

In [ ]:
def print_summary_table(master_df, ad_mci_patients, labels, enrichment_results):
    """
    Print final summary table: subtype, n_patients, clinical stats, top GO term.
    """
    print("\n" + "="*100)
    print("SUBTYPE SUMMARY TABLE")
    print("="*100)
    
    clinical_data = master_df.loc[ad_mci_patients].copy()
    clinical_data['subtype'] = labels
    
    k = len(np.unique(labels))
    
    for subtype in range(k):
        subtype_patients = clinical_data[clinical_data['subtype'] == subtype]
        n_patients = len(subtype_patients)
        mean_mmse = subtype_patients['mmse'].mean()
        mean_braak = subtype_patients['braaksc'].mean()
        mean_pseudo = subtype_patients['dpt_pseudotime'].mean()
        
        # Most abundant cell type
        ct_cols = ['ct_Ex', 'ct_In', 'ct_Ast', 'ct_Oli', 'ct_Mic', 'ct_OPCs']
        ct_labels = ['Ex', 'In', 'Ast', 'Oli', 'Mic', 'OPCs']
        mean_cts = subtype_patients[ct_cols].mean()
        dominant_ct = ct_labels[mean_cts.argmax()]
        
        # Top GO term
        top_go = "N/A"
        if subtype in enrichment_results and enrichment_results[subtype]:
            top_go = enrichment_results[subtype][0]['Term'][:50]
        
        print(f"\nSubtype {subtype+1} (ST{subtype+1}):")
        print(f"  Patients: {n_patients}")
        print(f"  Mean MMSE: {mean_mmse:.1f}")
        print(f"  Mean Braak: {mean_braak:.1f}")
        print(f"  Mean Pseudotime: {mean_pseudo:.3f}")
        print(f"  Dominant Cell Type: {dominant_ct}")
        print(f"  Top GO Term: {top_go}")
    
    print("\n" + "="*100)

## Main Pipeline

In [ ]:
print("\n" + "="*70)
print("STEP 1D: NMF CONSENSUS CLUSTERING FOR PATIENT STRATIFICATION")
print("="*70 + "\n")

try:
    # Load data
    master_df, bulk_df = load_data(PROCESSED_DATA_DIR)
    
    # Prepare clustering data
    clustering_matrix, ad_mci_patients, data_source = prepare_clustering_data(master_df, bulk_df, PROCESSED_DATA_DIR)
    
    # Ensure non-negative
    clustering_matrix = ensure_nonnegative(clustering_matrix)
    
    # Run consensus clustering
    results = run_nmf_consensus(clustering_matrix, k_range=K_RANGE, n_runs=N_CONSENSUS_RUNS)
    
    # Select optimal k
    optimal_k = select_optimal_k(results, cophenetic_threshold=COPHENETIC_THRESHOLD, min_cluster_size=MIN_CLUSTER_SIZE)
    
    # Plot selection
    plot_cluster_selection(results, RESULTS_DIR)
    
    # Get final labels
    final_labels = results[optimal_k]['consensus_labels']
    H_optimal = results[optimal_k]['H_base']
    
    # Get top proteins
    top_proteins_dict = get_top_proteins_per_subtype(clustering_matrix, H_optimal, top_n=TOP_PROTEINS_PER_SUBTYPE)
    
    # Run GO enrichment
    enrichment_results = run_go_enrichment(clustering_matrix, top_proteins_dict)
    
    # Clinical validation
    validation_results = clinical_validation(master_df, ad_mci_patients, final_labels, RESULTS_DIR)
    
    # Save results
    labels_df = save_subtype_labels(final_labels, ad_mci_patients, PROCESSED_DATA_DIR)
    master_final = create_master_final(master_df, labels_df, PROCESSED_DATA_DIR)
    
    # Plots
    plot_clinical_validation(master_df, ad_mci_patients, final_labels, RESULTS_DIR)
    plot_celltype_composition(master_df, ad_mci_patients, final_labels, RESULTS_DIR)
    
    # Summary
    print_summary_table(master_df, ad_mci_patients, final_labels, enrichment_results)
    
    print("\n" + "="*70)
    print("STEP 1D: NMF CLUSTERING COMPLETE")
    print("="*70)
    print(f"\nOutputs saved to: {PROCESSED_DATA_DIR}")
    print(f"Figures saved to: {RESULTS_DIR}")
    print("Ready for Step 1E: Pseudotime-stratified analysis\n")

except Exception as e:
    print(f"\nERROR: {str(e)}")
    import traceback
    traceback.print_exc()